# Load the Trained E1 ResNet50 Model — IP102

This notebook loads the existing **E1 Random Removal** checkpoint.

- Architecture: ResNet50
- Number of classes: 102
- Experiment: E1 — Random Removal
- Best epoch: 8
- Validation Macro-F1: 0.6221625840685715
- Seed: 42

The checkpoint is downloaded from GitHub Releases.

**No training or fine-tuning is performed.**


## 1. Clone the repository

In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path("/content/ip102-data-cleaning-research")

if not (repo_dir / ".git").exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/ecalban/ip102-data-cleaning-research.git",
            str(repo_dir),
        ],
        check=True,
    )
else:
    print("Repository already exists:", repo_dir)

assert (
    repo_dir / "results/E1/test_results/test_metrics.csv"
).is_file()

print("Repository ready:", repo_dir)


## 2. Download and verify the E1 checkpoint

In [ ]:
import hashlib
import requests

checkpoint_url = (
    "https://github.com/ecalban/ip102-data-cleaning-research/"
    "releases/download/e1-resnet50-v1/best_model.pth"
)

checkpoint_path = Path("/content/e1_best_model.pth")

expected_size = 285064472

expected_sha256 = (
    "7043a158aafedc5e17b25cb1756d0dfff3e59a163b256962f79708353dd1f818"
)

if not checkpoint_path.exists():
    with requests.get(
        checkpoint_url,
        stream=True,
        timeout=120,
        allow_redirects=True,
    ) as response:
        response.raise_for_status()

        with checkpoint_path.open("wb") as f:
            for chunk in response.iter_content(
                chunk_size=8 * 1024 * 1024
            ):
                if chunk:
                    f.write(chunk)

actual_size = checkpoint_path.stat().st_size

assert actual_size == expected_size, (
    f"File size mismatch: {actual_size}"
)

sha256 = hashlib.sha256()

with checkpoint_path.open("rb") as f:
    for chunk in iter(
        lambda: f.read(8 * 1024 * 1024),
        b"",
    ):
        sha256.update(chunk)

actual_sha256 = sha256.hexdigest()

assert actual_sha256 == expected_sha256, (
    "SHA-256 mismatch! This is not the verified E1 checkpoint."
)

print("E1 checkpoint downloaded and verified.")
print("Size:", actual_size, "bytes")
print("SHA-256:", actual_sha256)


## 3. Select CPU or GPU

In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 4. Load checkpoint metadata

In [ ]:
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

required_keys = {
    "epoch",
    "model_state_dict",
    "val_macro_f1",
}

assert required_keys.issubset(checkpoint.keys())

print("Epoch:", checkpoint["epoch"])
print("Validation Macro-F1:", checkpoint["val_macro_f1"])
print("Seed:", checkpoint.get("seed"))

assert checkpoint["epoch"] == 8
assert abs(
    checkpoint["val_macro_f1"] - 0.6221625840685715
) < 1e-10


## 5. Reconstruct ResNet50 and load E1 weights

In [ ]:
from torchvision import models
from torch import nn

# The original experiment used ImageNet-pretrained ResNet50.
# The checkpoint already contains the fine-tuned weights,
# so no separate ImageNet download is necessary.

model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 102)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

model = model.to(device)
model.eval()

print("E1 model loaded successfully.")
print("Output classes:", model.fc.out_features)
print("Training mode:", model.training)


## 6. Display original E1 test results

In [ ]:
import pandas as pd

results_dir = repo_dir / "results/E1/test_results"

test_metrics = pd.read_csv(
    results_dir / "test_metrics.csv"
)

print("Original E1 test metrics:")
display(test_metrics)

for filename in [
    "tail_recall.csv",
    "ece_result.csv",
]:
    path = results_dir / filename

    if path.exists():
        print(f"\\n{filename}")
        display(pd.read_csv(path))


## 7. Confirm inference readiness

In [ ]:
assert model.training is False
assert model.fc.out_features == 102

print("E1 ResNet50 is ready for inference.")
print("Device:", device)
print("No training was performed.")
